# Reduce words to their base form so that "feminists", "feminist", "feminism" all count as the same word.
What Chai says

Lemmatization is better than stemming because it produces real words

You need POS tags first — without them the lemmatizer makes mistakes

Example: "better" → "good", "leaves" → "leaf" (noun) or "leave" (verb)

#### 1. Imports
Using this: https://www.geeksforgeeks.org/python/python-lemmatization-with-nltk/

In [18]:
import pandas as pd
import ast
import nltk

nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


#### 2. Loading the Dataset

In [19]:
df_h1 = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/output_files/08_h1_stopwords.csv")
df_h1["tokens"] = df_h1["tokens"].apply(ast.literal_eval)

#### 3. Helper function to convert POS tags
Using this: https://www.geeksforgeeks.org/nlp/nlp-part-of-speech-default-tagging/

In [20]:
# NLTK uses its own POS tag format, but WordNet uses a different one
# this function converts between the two

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default to noun

#### 4. Writing the lemmatization function

In [21]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    
    # first get POS tag for each token
    pos_tags = nltk.pos_tag(tokens)
    
    # then lemmatize each token using its POS tag
    lemmatized = []
    for token, tag in pos_tags:
        wordnet_tag = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(token, wordnet_tag)
        lemmatized.append(lemma)
    
    return lemmatized

#### 5. Checking before and after in an example row

In [22]:
# check what lemmatization does to your actual text before applying to everything
sample = df_h1["tokens"].iloc[3]
print("BEFORE:", sample)
print("AFTER: ", lemmatize_tokens(sample))

BEFORE: ['women', 'aware', 'historically', 'connections', 'lives']
AFTER:  ['woman', 'aware', 'historically', 'connection', 'life']


#### 6. Applying the function

In [23]:
df_h1["tokens"] = df_h1["tokens"].apply(lemmatize_tokens)

#### 7. Checking

In [24]:
print(df_h1["tokens"].iloc[0])
print(f"H1 total tokens: {df_h1['tokens'].apply(len).sum()}")

['editorial', 'collective', 'lula', 'mae', 'blocton', 'yvonne', 'flower', 'valerie', 'harris', 'zarina', 'hashmi']
H1 total tokens: 52451


#### 8. Saving

In [25]:
df_h1.to_csv("/Users/sophiehamann/master-thesis-code/data/output_files/09_h1_lemmatized.csv", index=False)